In [1]:
# =============================================================================
# GUS01F: Loading Numerical Data onto GeoTERYT Records
# =============================================================================
# This notebook demonstrates the v4.0 data storage capabilities:
# 1. Load BDL demographic data (subject P2137 - population)
# 2. Process and attach time series to TERYTRecord objects
# 3. Query data on individual records
# 4. Aggregate for regions (voivodeships)
# 5. Produce joint/marginal distributions
# 6. Save/reload database with data persistence
# =============================================================================

# STEP 1: Imports and Path Setup
import os
import sys
from pathlib import Path
import importlib
import gc

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Find the repository root
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()
tools_path = repo_root / 'Code' / 'tools'
if str(tools_path) not in sys.path:
    sys.path.insert(0, str(tools_path))

# Reload geoTERYT_db to get latest version (v4.0)
import geoTERYT_db as gtdb
importlib.reload(gtdb)

# Data paths
data_root = repo_root.parent.parent / 'Data'
geo_root = data_root / 'Geospatial'
gus_root = data_root / 'GUS'

print(f"Repository root: {repo_root}")
print(f"Data root: {data_root}")
print(f"GUS root: {gus_root}")

Repository root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper
Data root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data
GUS root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS


In [2]:
# =============================================================================
# STEP 2: Load Complete GeoTERYT Database
# =============================================================================
complete_db_path = geo_root / 'geoteryt_complete_geom_OW.pkl'
db = gtdb.load_complete_database(complete_db_path)
db.print_summary()

Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_geom_OW.pkl...
  Database version: 3.1
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4560 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3612
  ✓ Records with old_woj: 2658
GeoTERYT Database Summary (v3.0)
Total records:           4,560
Year range:              1999 - 2024
------------------------------------------------------------
Administrative levels:
  Voivodeships (2):      16
  Powiats (5):           382
  Gminas (6):            4162
------------------------------------------------------------
Change tracking:
  Records with changes:      772
  Records with level changes: 0
  Records with kind changes:  0
------------------------------------------------------------
Geometry:
  Records with geometr

In [ ]:
# =============================================================================
# STEP 3: Load BDL Source Data
# =============================================================================
df_demographic = pd.read_csv(gus_root / "data" / 'bdl_demographic_data.csv', encoding='utf-8')
df_variables = pd.read_csv(gus_root / "metadata" / 'bdl_variables_level6.csv', encoding='utf-8')

print(f"df_demographic: {df_demographic.shape}")
print(f"df_variables: {df_variables.shape}")
print(f"\nAvailable subjects: {sorted(df_demographic['subjectId'].unique())}")

In [ ]:
# =============================================================================
# STEP 4: Process Subject P2137 (Population Data)
# =============================================================================
# Use the new process_subject_data() static method on GeoTERYTDatabase
subject_id = 'P2137'
df_p2137 = gtdb.GeoTERYTDatabase.process_subject_data(df_demographic, df_variables, subject_id)

print(f"Processed P2137: {df_p2137.shape}")
print(f"\nColumns: {list(df_p2137.columns)}")
print(f"\nCategory columns present:")
for col in ['n1', 'n2', 'n3', 'n4', 'n5']:
    if col in df_p2137.columns:
        print(f"  {col}: {sorted(df_p2137[col].dropna().unique())}")
print(f"\nYears: {sorted(df_p2137['year'].dropna().astype(str).unique())}")
print(f"Unique TERYT IDs: {df_p2137['teryt_id'].nunique()}")
df_p2137.head()

In [ ]:
# =============================================================================
# STEP 5: Load Subject Data onto TERYTRecords
# =============================================================================
# This attaches time series data to each matching TERYTRecord in the database
stats = db.load_subject_data(df_p2137, source_type='BDL', subject_id='P2137')
print(f"\nLoading statistics: {stats['matched_teryts']} matched, {stats['unmatched_teryts']} unmatched")
print(f"Total data points loaded: {stats['total_data_points']:,}")

# Free memory
del df_p2137
gc.collect()

In [ ]:
# =============================================================================
# STEP 6: Verify Data on Individual Records
# =============================================================================
# Check data summary
data_summary = db.get_data_summary()
print("Data Summary:")
for k, v in data_summary.items():
    print(f"  {k}: {v}")

# Look at a specific record (e.g., Kraków)
print("\n--- Example: Kraków ---")
krakow_records = [r for r in db._records.values() if 'Kraków' in r.name and r.level == 6]
if krakow_records:
    rec = krakow_records[0]
    print(f"Record: {rec}")
    print(f"Data keys: {rec.list_data_keys()[:5]}...")
    print(f"Total data series: {rec.n_data_series}")
    
    # Show one time series
    if rec.data:
        first_key = list(rec.data.keys())[0]
        series = rec.data[first_key]
        print(f"\nFirst series: {series}")
        print(f"  Years: {series.years[:5]}...{series.years[-3:]}")
        print(f"  Value in 2020: {series.get_value(2020)}")
        print(f"  Categories: {series.categories}")

In [ ]:
# =============================================================================
# STEP 7: Aggregate Data for a Voivodeship (Regional Totals)
# =============================================================================
# Get all gminas in Małopolskie (voivodeship code '12') for year 2020
malopolskie_gminas = db.get_gminas_in_voivodeship('12', year=2020)
print(f"Małopolskie gminas in 2020: {len(malopolskie_gminas)}")

# Aggregate all population variables for the voivodeship
agg_df = db.aggregate_data(malopolskie_gminas, 'P2137', 2020, agg_func='sum')
print(f"\nAggregated data shape: {agg_df.shape}")
print(f"Columns: {list(agg_df.columns)}")
agg_df

In [ ]:
# =============================================================================
# STEP 8: Joint and Marginal Distributions
# =============================================================================
# Joint distribution: age group (n1) × gender (n2) for Małopolskie, year 2020
print("=== Joint Distribution (age group × gender) - Małopolskie 2020 ===")
joint = db.get_distribution(malopolskie_gminas, 'P2137', 2020,
                            row_category='n1', col_category='n2')
display(joint)

# Marginal distribution: by gender only
print("\n=== Marginal Distribution (by gender) - Małopolskie 2020 ===")
marginal_gender = db.get_distribution(malopolskie_gminas, 'P2137', 2020,
                                       col_category='n2')
display(marginal_gender)

# Marginal distribution: by age group only
print("\n=== Marginal Distribution (by age group) - Małopolskie 2020 ===")
marginal_age = db.get_distribution(malopolskie_gminas, 'P2137', 2020,
                                    row_category='n1')
display(marginal_age)

In [ ]:
# =============================================================================
# STEP 9: Save Database with Data and Verify Persistence
# =============================================================================
# Save the database with data attached
save_path = geo_root / 'geoteryt_complete_final.pkl'
db.save_complete(save_path)

# Reload and verify
db2 = gtdb.load_complete_database(save_path)
db2.print_summary()

# Verify data survived the save/load cycle
summary2 = db2.get_data_summary()
print(f"\nData after reload: {summary2['records_with_data']} records with data, "
      f"{summary2['total_data_points']:,} total points")

# Quick sanity check: same Kraków record
krakow2 = [r for r in db2._records.values() if 'Kraków' in r.name and r.level == 6]
if krakow2:
    print(f"\nKraków data series after reload: {krakow2[0].n_data_series}")
    first_key = list(krakow2[0].data.keys())[0]
    print(f"First series: {krakow2[0].data[first_key]}")

del db2
gc.collect()